# VOS → Wan2.2 Animate Replacement (sem API de inferência)
O Colab gratuito T4 16 GB NÃO foi validado. O diagnóstico deve passar antes de baixar pesos. Uma cena por ZIP, quadros originais.

In [ ]:
import sys, pathlib, shutil, subprocess
from google.colab import files
print('Espaco livre:',round(shutil.disk_usage('/content').free/2**30,2),'GiB')

## 0. Diagnóstico rápido, ANTES de enviar ZIP ou baixar pesos
Execute esta célula primeiro, já com uma GPU gratuita selecionada. Ela gera `VOS_GPU_DIAGNOSTICO.json`. Se bloquear, copie o resultado e interrompa; **não há necessidade de enviar seu vídeo**. Os 32/24/85 GiB são limites preventivos, não garantia de execução.

In [ ]:
def check_resources():
    """Diagnostico leve: roda antes de uploads/instalacao, nao baixa modelos."""
    import torch, psutil, shutil, json
    from pathlib import Path
    from datetime import datetime, timezone
    gpu_present = torch.cuda.is_available()
    p = torch.cuda.get_device_properties(0) if gpu_present else None
    vram = p.total_memory / 2**30 if p else 0
    ram = psutil.virtual_memory().available / 2**30
    disk = shutil.disk_usage('/content').free / 2**30
    bf16 = bool(torch.cuda.is_bf16_supported()) if gpu_present else False
    report = {
        'timestamp_utc':datetime.now(timezone.utc).isoformat(),
        'gpu':p.name if p else 'CUDA indisponivel',
        'vram_gib':round(vram,2),
        'available_ram_gib':round(ram,2),
        'free_disk_gib':round(disk,2),
        'bf16_nativo':bf16,
        'passou_bloqueio_preventivo':gpu_present and vram>=32 and ram>=24 and disk>=85 and bf16,
        'observacao':'Criterios conservadores do notebook; passar nao garante Wan 14B'
    }
    print('=== DIAGNOSTICO VOS/WAN, SEM DOWNLOADS ===')
    print(json.dumps(report, ensure_ascii=False, indent=2))
    try:
        if Path('/content').is_dir():
            Path('/content/VOS_GPU_DIAGNOSTICO.json').write_text(
                json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
            print('Relatorio salvo: /content/VOS_GPU_DIAGNOSTICO.json')
    except OSError as error:
        print('Relatorio nao salvo:',error)
    if not gpu_present:
        raise RuntimeError('CUDA indisponivel no Colab gratuito. Nao envie ZIP nem baixe pesos.')
    if vram<32 or ram<24 or disk<85:
        raise RuntimeError('Recursos insuficientes para tentativa conservadora Wan Animate 14B. Nenhuma inferencia foi executada.')
    if not bf16:
        raise RuntimeError('GPU sem BF16 nativo para esta configuracao Wan')
    return report
try:
    check_resources()
except BaseException:
    import traceback,zipfile,pathlib
    pathlib.Path('VOS_PREFLIGHT_FALHA.log').write_text(traceback.format_exc())
    with zipfile.ZipFile('VOS_PREFLIGHT_LOGS.zip','w') as logs:
        for name in ('VOS_GPU_DIAGNOSTICO.json','VOS_PREFLIGHT_FALHA.log'):
            if pathlib.Path(name).exists():logs.write(name)
    print('GPU/RAM/disco insuficientes: nao envie video nem baixe pesos. Guarde VOS_PREFLIGHT_LOGS.zip pelo painel Arquivos.')
    raise
print('Diagnostico passou: agora envie os arquivos; isto NAO garante geracao.')


## 1. Enviar ZIP VOS (uma cena, FPS original) e imagem do novo personagem.

In [ ]:
print('Escolha ZIP do VOS')
u=files.upload()
ZIP=pathlib.Path('/content')/next(n for n in u if n.lower().endswith('.zip'))
print('Escolha imagem do novo personagem')
u=files.upload()
REF=pathlib.Path('/content')/next(n for n in u if n.lower().endswith(('.png','.jpg','.jpeg')))
print('Selecionados:',ZIP,REF)
del u  # libera os bytes do upload apos salvar os arquivos

## 2. Preparar cena (sem IA). Baixa somente o script do seu repositório GitHub.

In [ ]:
import urllib.request
VOS_REF='codex/finalize-wan22-validation'  # para reproduzir exatamente, use o SHA revisado do PR
TOOLS=pathlib.Path('/content/vos_tools/frame_studio')
(TOOLS/'static').mkdir(parents=True,exist_ok=True)
for name in ('__init__.py','wan_bridge.py','target_masks.py','wan_target.py','colab_support.py','static/target_editor.js'):
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/aruanmf446-hub/Vos/{VOS_REF}/frame_studio/{name}',str(TOOLS/name))
sys.path.insert(0,str(TOOLS.parent))
from frame_studio.colab_support import Session,install_error_hook
WORK=pathlib.Path('/content/vos_wan_job')
SESSION=Session('/content/vos_logs',work=WORK,download=files.download)
install_error_hook(SESSION,get_ipython())
# Inclui o diagnostico zero-download no pacote de evidencias.
diagnostic=pathlib.Path('VOS_GPU_DIAGNOSTICO.json')
if diagnostic.exists():shutil.copyfile(diagnostic,SESSION.folder/diagnostic.name)
BRIDGE=TOOLS/'wan_bridge.py'
SESSION.run([sys.executable,str(BRIDGE),'prepare','--zip',str(ZIP),'--reference',str(REF),'--work',str(WORK)],label='Preparar ZIP original')

In [ ]:
from IPython.display import display, Video
display(Video(str(WORK/'driving.mp4'),embed=True))

## Opcional — selecionar exatamente o personagem (experimental, desligado por padrão)
Ative somente para testar a seleção explícita. No editor: pontos verdes no alvo (ex.: Wanda), vermelhos na outra pessoa (ex.: Estranho). Adicione pontos em quadros-chave quando houver cruzamento/oclusão. Nada é enviado para outro serviço. O editor abre nesta célula; clique em Salvar para continuar.


In [ ]:
USE_TARGET_SELECTION=False #@param {type:"boolean"}
from frame_studio.target_masks import show_editor
if USE_TARGET_SELECTION:
    show_editor(WORK,'prompts')
elif (WORK/'target.json').exists():
    raise RuntimeError('Selecao ja ativa neste trabalho. Use um trabalho novo para voltar ao modo automatico; nao apague os recibos manualmente.')

## 3. Instalar Wan oficial e baixar pesos via ModelScope, jamais Hugging Face. Download de arquivos via internet; não é API de inferência.

In [ ]:
check_resources()  # bloqueia instalacao mesmo se a celula anterior falhou
import os
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_DATASETS_OFFLINE='1')
WAN_REV='1ea34ff48f87168174e12956e200b1d908b1c5ff'
WAN=pathlib.Path('/content/Wan2.2')
if not WAN.exists():SESSION.run(['git','clone','--no-checkout','https://github.com/Wan-Video/Wan2.2.git',str(WAN)])
SESSION.run(['git','-C',str(WAN),'checkout','--detach',WAN_REV])
SESSION.run([sys.executable,'-m','pip','install','--no-build-isolation','-r',str(WAN/'requirements.txt'),'-r',str(WAN/'requirements_animate.txt'),'moviepy==1.0.3','diffusers==0.35.1'])
SESSION.run([sys.executable,'-m','pip','install','modelscope'])
CKPT=pathlib.Path('/content/Wan2.2-Animate-14B')
WEIGHTS_MANIFEST=pathlib.Path('/content/wan22_modelscope_manifest.json')
urllib.request.urlretrieve(f'https://raw.githubusercontent.com/aruanmf446-hub/Vos/{VOS_REF}/colab/wan22_modelscope_manifest.json',str(WEIGHTS_MANIFEST))
import json
weights=json.loads(WEIGHTS_MANIFEST.read_text())
if shutil.disk_usage('/content').free < sum(entry['size'] for entry in weights['files'])+15*2**30:
    raise RuntimeError('Disco insuficiente para pesos selecionados e arquivos de trabalho')
from modelscope import snapshot_download
snapshot_download(weights['model_id'],revision=weights['revision'],local_dir=str(CKPT),allow_patterns=[entry['path'] for entry in weights['files']])
SESSION.run([sys.executable,str(BRIDGE),'verify-weights','--checkpoint',str(CKPT),'--manifest',str(WEIGHTS_MANIFEST)])
print('Wan revision:',WAN_REV)
SESSION.run([sys.executable,'-m','pip','freeze'])

## Opcional — SAM2 e revisão nativa antes do Wan
Esta etapa só roda se a opção acima foi ativada. O SAM2 usa os pontos positivos/negativos e os quadros-chave corrigidos; não foi comprovado neste ambiente sem GPU. A sobreposição vermelha indica a área alterada. Use pincel/borracha para corrigir um quadro e marque cada quadro como revisado. Trocas/saltos suspeitos bloqueiam a aprovação. Se o editor pedir novo rastreamento, execute esta célula novamente; as correções salvas servem de quadros-chave ao SAM2. Não há interpolação de máscaras.


In [ ]:
if USE_TARGET_SELECTION:
    SESSION.run([sys.executable,str(TOOLS/'wan_target.py'),'track','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)],label='SAM2 alvo explicito (experimental)')
    show_editor(WORK,'review')
    from frame_studio.target_masks import export_review
    files.download(str(export_review(WORK,WORK.parent/'VOS_MASCARAS_REVISAO.zip')))
else:
    print('Modo atual preservado: selecao automatica oficial, exige revisao visual.')

## 4. Pré-processar a cena sem iniciar o modelo generativo
O Wan oficial prepara pose, rosto, cenário e máscara do personagem. Uma verificação bloqueia máscaras vazias, enormes ou com contagem de quadros divergente.

In [ ]:
SESSION.run([sys.executable,str(BRIDGE),'preprocess','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)])
print('Pré-processamento completo. Revise os vídeos abaixo ANTES de prosseguir.')

## 5. Revisão obrigatória da máscara (principalmente se houver outras pessoas)
Compare a cena de condução, a máscara (branco = área alterada) e os movimentos da pose. Se marcar pessoa errada, interrompa. O pré-processamento automático do Wan foi projetado para vídeo com **uma pessoa**; não garante preservação de múltiplos atores.

In [ ]:
(WORK/'mask_review.json').unlink(missing_ok=True)
from IPython.display import display,Video
for n in ('driving.mp4','wan_processed/src_mask.mp4','wan_processed/src_pose.mp4'):
    print('Revisar:',n)
    display(Video(str(WORK/n),embed=True))
confirmacao=input('Mascara segue APENAS o personagem correto em todos os frames? Digite SIM: ')
if confirmacao.strip().upper()!='SIM':raise RuntimeError('Geração cancelada: revise a máscara e a cena no VOS')
SESSION.run([sys.executable,str(BRIDGE),'review-mask','--work',str(WORK),'--confirm','SIM'])

## 6. Executar Wan2.2 Animate em modo Replacement
Esta etapa demanda GPU real e checkpoints oficiais. A prévia de máscara não é uma geração concluída. Se ocorrer OOM, não há resultado.

In [ ]:
LOW_MEMORY_EXPERIMENTAL=False #@param {type:"boolean"}
# Flags oficiais: T5 na CPU e janelas de 33 quadros (padrao Wan: 77).
# Podem aumentar uso de RAM/tempo e mudar continuidade entre janelas. Sem garantia T4.
command=[sys.executable,str(BRIDGE),'infer','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)]
if LOW_MEMORY_EXPERIMENTAL:command.append('--low-memory')
SESSION.run(command,label='Wan Animate Replacement real')
display(Video(str(WORK/'wan_generated.mp4'),embed=True))

## 7. Montar ZIP editado válido para VOS e baixar
Se a geração não produziu quadros na contagem esperada, a montagem aborta.

In [ ]:
OUT=pathlib.Path('/content/VOS_EDITADO.zip')
SESSION.run([sys.executable,str(BRIDGE),'finish','--work',str(WORK),'--output',str(OUT)])
files.download(str(OUT))
files.download(str(SESSION.bundle()))

## 8. Voltar ao VOS no Render
No **mesmo projeto e sessão**: Importar edições > Anexar VOS_EDITADO.zip > Validar > Comparar > Aplicar pacote > Montar vídeo (FPS original).